In [1]:
from langchain_groq import ChatGroq
# from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
# from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
# import google.generativeai as genai
import torch
import os

load_dotenv()

True

In [2]:
## Task1: Prompt Template

# 1. Create Prompt Template with system instruction and user question

prompt = PromptTemplate(
    template="System: You are a helpful assistant. Answer the user's question in maximum 50 words. \nUser Question: {data}",
    input_variables=["data"]
)

## Create Model
model = ChatGroq(model="openai/gpt-oss-120b",temperature=0)


## Inject user input dynamically\
question =  "What is Python ?"
formatted_prompt = prompt.format(data=question)

formatted_prompt

## Get Output

output = model.invoke(formatted_prompt)
output

## Test with Multiple Inputs
questions = ["what is Python?", "What is Machine Learning", "What is LangChain"]


for question in questions:
    formatted_prompt = prompt.format(data = question)
    output = model.invoke(formatted_prompt)
    print("\nQuestion:",question)
    print("\nAnswer:", output.content)
  


Question: what is Python?

Answer: Python is a high‑level, interpreted programming language known for its readability and simplicity. It supports multiple paradigms (procedural, object‑oriented, functional) and has a vast standard library, making it popular for web development, data science, automation, and more.

Question: What is Machine Learning

Answer: Machine learning is a subset of AI where computers learn patterns from data to make predictions or decisions without explicit programming. Algorithms build models from examples, improving performance as more data is provided.

Question: What is LangChain

Answer: LangChain is an open‑source framework for building applications that combine large language models (LLMs) with external data sources, APIs, and custom logic, enabling developers to create conversational agents and pipelines that retrieve, reason, and act on information.


In [3]:
## Task 2: ChatPromptTemplate and Message Template



In [4]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_groq import ChatGroq
import torch

## Creating Chat Template
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer in maximum 50 words"),
    ("human" , " {question}"),
    ('ai', "I will answer the user's question clearly")
])

chat_prompt

## Creating Model
model = ChatGroq(model="openai/gpt-oss-120b",temperature=0)

## Inject user input dynamically
question = "What is genai"
formatted_prompt = chat_prompt.format_messages(question = question )

## Output
output = model.invoke(formatted_prompt)
output.content


## Comapare with Simple PromptTemplate
## ChatPromptTemplate supports System, Human, and AI message roles to structure a conversation, whereas PromptTemplate creates a single text prompt without separating these message roles.


'GenAI (generative AI) is artificial‑intelligence software that creates new content—text, images, audio, video, code, etc.—instead of just analyzing data. It learns patterns from large datasets and then produces original outputs, powering tools like ChatGPT, DALL·E, and music‑generation models.'

In [5]:
## Part 2: Structured Output using Pydantic
   ## Task 3 : Pydantic Output Schema

from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser


## Define Pydantic Model
class Answer(BaseModel):
    answer:str
    confidence:float
    source: str


## create Output Parser
parser = PydanticOutputParser(pydantic_object=Answer)

## Create Prompt Template
prompt = PromptTemplate(template='''Answer the user's question.\n{format_instructions}\nQuestion: {question}''', input_variable=['format_instructions','questions'], partial_variables={"format_instructions":parser.get_format_instructions()})


## Creating Model
model = ChatGroq(model="openai/gpt-oss-120b",temperature=0)

## Get User Input
question = "Why we use Python"

formatted_prompt = prompt.format(question = question)

## Get Output and Parse It
output = model.invoke(formatted_prompt)
result = parser.parse(output.content)

print("Answer:", result.answer)
print("Answer Data Type:", type(result.answer))

print("Confidence:", result.confidence)
print("Confidence Data Type:", type(result.confidence))

print("Source:", result.source)
print("Source Data Type:", type(result.source))



Answer: Python is popular because it has a simple, readable syntax that makes it easy to learn and write, a vast standard library and ecosystem of third‑party packages for tasks ranging from web development to data science, strong cross‑platform support, and an active community that provides extensive documentation and support.
Answer Data Type: <class 'str'>
Confidence: 0.97
Confidence Data Type: <class 'float'>
Source: Python official documentation and common programming resources
Source Data Type: <class 'str'>


In [6]:
## Part 2: Structured Output using Pydantic
   ## Task 4 : Validation and Error Handling

from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser


## Define Pydantic Model
class Answer(BaseModel):
    answer:str
    confidence:float
    source: str


## create Output Parser
parser = PydanticOutputParser(pydantic_object=Answer)

## Create Prompt Template
prompt = PromptTemplate(template='''Answer the user's question.\n{format_instructions}\nQuestion: {question}''', input_variable=['format_instructions','questions'], partial_variables={"format_instructions":parser.get_format_instructions()})


## Creating Model
model = ChatGroq(model="openai/gpt-oss-120b",temperature=0)

## Get User Input
question = "Why we use Python"

formatted_prompt = prompt.format(question = question)

## Get Output and Parse It
# output = model.invoke(formatted_prompt)
# result = parser.parse(output.content)

# print("Answer:", result.answer)
# print("Answer Data Type:", type(result.answer))

# print("Confidence:", result.confidence)
# print("Confidence Data Type:", type(result.confidence))

# print("Source:", result.source)
# print("Source Data Type:", type(result.source))


try:
    output = model.invoke(formatted_prompt)
    result = parser.parse(output.content)

    print("Answer:", result.answer)
    print("Confidence:", result.confidence)
    print("Source:", result.source)

except Exception as e:
    print("Error: Invalid LLM output")
    print(e)

    print("Error: Invalid LLM output")
    print("Details:", e)

     # Fallback response
    print("Fallback: Unable to generate a valid response.")



Answer: Python is popular because it has a simple, readable syntax that makes it easy to learn and write, supports multiple programming paradigms (procedural, object‑oriented, functional), runs on all major platforms, and offers a vast standard library plus thousands of third‑party packages for tasks ranging from web development to data science, machine learning, automation, and scientific computing. Its strong community, rapid development cycle, and excellent integration with other languages and tools also make it a versatile choice for both beginners and experienced developers.
Confidence: 0.97
Source: Python official documentation and common programming resources
